In [ ]:
import pandas as pd

In [ ]:
def clean_data_follow_ups(df):
    # Convert missing values represented as "'--" to None type
    df = df.replace("'--", None)
    # Remove columns with only one distinct value
    df = df.loc[:, df.nunique() > 1]

    # Remove rows for molecular data
    # Filter rows based on column: 'molecular_tests.submitter_id'
    df = df[df['molecular_tests.submitter_id'].isna()]

    # Remove empty columns for molecular data
    # Drop columns: 'molecular_tests.test_result', 'molecular_tests.gene_symbol' and 3 other columns
    df = df.drop(columns=['molecular_tests.test_result', 'molecular_tests.gene_symbol', 'molecular_tests.molecular_test_id', 'molecular_tests.ploidy', 'molecular_tests.submitter_id'])


    # Remove redundant column, days to first event
    # Drop column: 'follow_ups.days_to_first_event'
    df = df.drop(columns=['follow_ups.days_to_first_event'])

    # Remove column: year of follow up
    # Drop column: 'follow_ups.year_of_follow_up'
    df = df.drop(columns=['follow_ups.year_of_follow_up'])

    # Reorder "label" columns
    col = df.pop("follow_ups.days_to_follow_up")
    df.insert(len(df.columns), "follow_ups.days_to_follow_up", col)
    col = df.pop("follow_ups.first_event")
    df.insert(len(df.columns), "follow_ups.first_event", col)

    # Sort by column: 'follow_ups.first_event' (ascending)
    df = df.sort_values(['follow_ups.first_event'])
    # Remove duplicate days to follow up
    df = df.drop_duplicates(subset=["cases.case_id", "follow_ups.days_to_follow_up"], keep="first")
    # Sort by case_id
    df = df.sort_values(['cases.case_id'])

    # Change column type to float32 for columns: 'follow_ups.days_to_first_event', 'follow_ups.days_to_follow_up'
    df = df.astype({'follow_ups.days_to_follow_up': 'float32'})

    # Remove extra ID columns
    df = df.drop(columns=['cases.submitter_id', 'follow_ups.follow_up_id', 'follow_ups.submitter_id'])

    # Drop column: 'follow_ups.progression_or_recurrence_anatomic_site'
    df = df.drop(columns=['follow_ups.progression_or_recurrence_anatomic_site'])

    # For cases with relapse, remove follow ups that occur after the relapse event
    relapse_rows = df.loc[df["follow_ups.first_event"] == "Relapse"]
    temp_relapse_days = relapse_rows.groupby('cases.case_id')['follow_ups.days_to_follow_up'].min().rename('temp_relapse_days')
    # add in new temp column
    df = df.merge(temp_relapse_days, on="cases.case_id", how="left")
    # keep rows where there is no relapse OR the days to follow up is <= the days to relapse, if applicable
    df = df[df['temp_relapse_days'].isna() | (df['follow_ups.days_to_follow_up'] <= df['temp_relapse_days'])]
    # Remove temp column
    df = df.drop(columns=['temp_relapse_days'])

    # Drop column: 'follow_ups.timepoint_category'
    df = df.drop(columns=['follow_ups.timepoint_category'])

    # One-hot encode first_event
    first_event_one_hot = pd.get_dummies(df, columns=["follow_ups.first_event"], dtype=float)
    df = pd.concat([first_event_one_hot], axis=1)

    # Simplifying the problem, removing other types of events
    df = df.drop(columns=['follow_ups.first_event_Progression', 'follow_ups.first_event_Event', 'follow_ups.first_event_Death', 'follow_ups.first_event_Censored', 'follow_ups.first_event_Second Malignant Neoplasm'])

    return df

df_follow_ups_A = pd.read_csv("dataset/follow_up.tsv", sep='\t')

df_clean_follow_ups = clean_data_follow_ups(df_follow_ups_A.copy())
print(df_clean_follow_ups.shape)
df_clean_follow_ups.head(15)

In [ ]:
def clean_data_molecular(df):
    # Convert missing values represented as "'--" to None type
    df = df.replace("'--", None)
    # Remove columns with only one distinct value
    df = df.loc[:, df.nunique() > 1]

    # Isolate rows containing molecular data
    # Filter rows based on column: 'molecular_tests.submitter_id'
    df = df[df['molecular_tests.submitter_id'].notna()]

    # Remove empty columns
    # Drop columns: 'follow_ups.days_to_first_event', 'follow_ups.days_to_follow_up' and 5 other columns
    df = df.drop(columns=['follow_ups.days_to_first_event', 'follow_ups.days_to_follow_up', 'follow_ups.first_event', 'follow_ups.progression_or_recurrence_anatomic_site', 'follow_ups.submitter_id', 'follow_ups.timepoint_category', 'follow_ups.year_of_follow_up'])
    col = df.pop("molecular_tests.gene_symbol")
    df.insert(len(df.columns), "molecular_tests.gene_symbol", col)
    col = df.pop("molecular_tests.ploidy")
    df.insert(len(df.columns), "molecular_tests.ploidy", col)
    col = df.pop("molecular_tests.test_result")
    df.insert(len(df.columns), "molecular_tests.test_result", col)

    # Drop column: 'molecular_tests.gene_symbol'
    # In this dataset focused on neuroblastoma, MYCN is the sole gene symbol, so it's not really informative here
    df = df.drop(columns=['molecular_tests.gene_symbol'])

    # Consider values marked Unknown as missing
    df = df.replace("Unknown", None)

    # Drop extra ID columns
    # Drop columns: 'cases.submitter_id', 'follow_ups.follow_up_id' and 2 other columns
    df = df.drop(columns=['cases.submitter_id', 'follow_ups.follow_up_id', 'molecular_tests.molecular_test_id', 'molecular_tests.submitter_id'])

    # One-hot encode ploidy
    ploidy_one_hot = pd.get_dummies(df, dummy_na=True, columns=["molecular_tests.ploidy"], dtype=int)
    df = pd.concat([ploidy_one_hot], axis=1)

    # One-hot encode test result
    test_result_one_hot = pd.get_dummies(df, dummy_na=True, columns=["molecular_tests.test_result"], dtype=int)
    df = pd.concat([test_result_one_hot], axis=1)

    # Collapse rows by case_id and keep values for each column accordingly
    df = df.groupby("cases.case_id").agg({
    "molecular_tests.ploidy_Diploid":"max",
    "molecular_tests.ploidy_Hyperdiploid":"max",
    "molecular_tests.ploidy_nan":"min",

    "molecular_tests.test_result_Abnormal, NOS":"max",
    "molecular_tests.test_result_Amplified":"max",
    "molecular_tests.test_result_Normal":"max",
    "molecular_tests.test_result_Not Amplified":"max",
    "molecular_tests.test_result_nan":"min"
    })

    df = df.reset_index()
    
    return df

df = pd.read_csv("dataset/follow_up.tsv", sep='\t')

df_clean_molecular = clean_data_molecular(df.copy())
print(df_clean_molecular.shape)
df_clean_molecular.head(15)

In [ ]:
def clean_data_pathology(df):
    # Convert missing values represented as "'--" to None type
    df = df.replace("'--", None)
    # Remove columns with only one distinct value
    df = df.loc[:, df.nunique() > 1]

    # Remove extra ID columns
    df = df.drop(columns=['pathology_details.pathology_detail_id', 'pathology_details.submitter_id', 'diagnoses.submitter_id', 'diagnoses.diagnosis_id', 'cases.submitter_id'])
    
    # Change column type to float32 for column: 'pathology_details.necrosis_percent'
    df = df.astype({'pathology_details.necrosis_percent': 'float32'})

    # Change column type to float32 for column: 'pathology_details.percent_tumor_nuclei'
    df = df.astype({'pathology_details.percent_tumor_nuclei': 'float32'})
    return df

df = pd.read_csv("dataset/pathology_detail.tsv", sep='\t')

df_clean_pathology = clean_data_pathology(df.copy())
df_clean_pathology.head()

In [ ]:
def clean_data_clinical(df):
    # Convert missing values represented as "'--" to None type
    df = df.replace("'--", None)
    # Remove columns with only one distinct value
    df = df.loc[:, df.nunique() > 1]

    # Remove rows missing more than half of columns
    df = df.dropna(thresh=len(df.columns)/2)

    # Remove disease type column since all data is from the same disease
    df = df.drop(columns=['cases.disease_type'])

    # Remove extra ID fields
    df = df.drop(columns=['treatments.treatment_id', 'treatments.submitter_id', 'diagnoses.submitter_id', 'diagnoses.diagnosis_id', 'demographic.submitter_id', 'demographic.demographic_id', 'cases.submitter_id'])

    # Remove age by year field
    df = df.drop(columns=['demographic.age_at_index'])

    # Remove days to birth field, redundant 
    df = df.drop(columns=['demographic.days_to_birth'])

    # Remove year of diagnosis field
    df = df.drop(columns=['diagnoses.year_of_diagnosis'])
    
    return df

df = pd.read_csv("dataset/clinical.tsv", sep='\t')

df_clean_clinical = clean_data_clinical(df.copy())
df_clean_clinical.head(15)